### PRN = 8057
### Name = Yugal Kaushal

In [38]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer,make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

In [39]:
from ISLP import load_data
Bikeshare = load_data('Bikeshare')
Bikeshare.columns

Index(['season', 'mnth', 'day', 'hr', 'holiday', 'weekday', 'workingday',
       'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'casual',
       'registered', 'bikers'],
      dtype='object')

In [40]:
Bikeshare.head()

,season,mnth,day,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,bikers
0,1,Jan,1,0,0,6,0,clear,0.24,0.2879,0.81,0.0,3,13,16
1,1,Jan,1,1,0,6,0,clear,0.22,0.2727,0.80,0.0,8,32,40
2,1,Jan,1,2,0,6,0,clear,0.22,0.2727,0.80,0.0,5,27,32
3,1,Jan,1,3,0,6,0,clear,0.24,0.2879,0.75,0.0,3,10,13
4,1,Jan,1,4,0,6,0,clear,0.24,0.2879,0.75,0.0,0,1,1


In [41]:
Bikeshare.isnull().sum().sum()

np.int64(0)

In [42]:
X,y=Bikeshare.drop('bikers',axis=1),Bikeshare['bikers']

In [43]:
ohe = OneHotEncoder(sparse_output=False,drop='first')
ct = make_column_transformer((ohe,make_column_selector(dtype_include=object)),
                            ('passthrough',make_column_selector(dtype_include=['int64','float64'])),
                            verbose_feature_names_out=False).set_output(transform='pandas')
kfold = KFold(random_state=26, n_splits=5, shuffle=True)

# ElasticNet

In [44]:
en = ElasticNet()

In [45]:
pipeEN = Pipeline([("CT",ct),('EN',en)])

In [46]:
pipeEN.get_params()

{'memory': None,
 'steps': [('CT',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000014B8990EAD0>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000014B8990ECF0>)],
                     verbose_feature_names_out=False)),
  ('EN', ElasticNet())],
 'transform_input': None,
 'verbose': False,
 'CT': ColumnTransformer(transformers=[('onehotencoder',
                                  OneHotEncoder(drop='first',
                                                sparse_output=False),
                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000014B8990EAD0>),
                  

In [49]:
params = {'EN__alpha':np.linspace(0.001,10,20),
          'EN__max_iter': [10000],
          'EN__random_state':[26]
         }

In [50]:
gridCV = GridSearchCV(pipeEN,param_grid=params,scoring='r2',cv=kfold)
gridCV.fit(X,y)
print(f'Best Params : {gridCV.best_params_}')
print(f'Best Score : {gridCV.best_score_}')

Best Params : {'EN__alpha': np.float64(0.001), 'EN__max_iter': 10000, 'EN__random_state': 26}
Best Score : 0.9999999999998904


In [52]:
random_gridCV = RandomizedSearchCV(pipeEN,param_distributions=params,scoring='r2',cv=kfold)
random_gridCV.fit(X,y)
print(f'Best Params : {random_gridCV.best_params_}')
print(f'Best Score : {random_gridCV.best_score_}')

Best Params : {'EN__random_state': 26, 'EN__max_iter': 10000, 'EN__alpha': np.float64(0.001)}
Best Score : 0.9999999999998904


# GradientBoostingRegressor

In [53]:
gb = GradientBoostingRegressor()

In [55]:
pipeGB = Pipeline([("CT",ct),('GB',gb)])
pipeGB.get_params()

{'memory': None,
 'steps': [('CT',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000014B8990EAD0>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000014B8990ECF0>)],
                     verbose_feature_names_out=False)),
  ('GB', GradientBoostingRegressor())],
 'transform_input': None,
 'verbose': False,
 'CT': ColumnTransformer(transformers=[('onehotencoder',
                                  OneHotEncoder(drop='first',
                                                sparse_output=False),
                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000014B8990EAD0>),
   

In [60]:
params = {'GB__learning_rate':np.linspace(0.01,0.9,10),
          'GB__n_estimators':[25,50,100],
          'GB__max_depth': [None,2,3,4],
          'GB__random_state':[26]
         }

In [62]:
gridCV = GridSearchCV(pipeGB,param_grid=params,scoring='r2',cv=kfold,n_jobs=-1)
gridCV.fit(X,y)
print(f'Best Params : {gridCV.best_params_}')
print(f'Best Score : {gridCV.best_score_}')

Best Params : {'GB__learning_rate': np.float64(0.10888888888888888), 'GB__max_depth': 4, 'GB__n_estimators': 100, 'GB__random_state': 26}
Best Score : 0.9995614049446825


In [61]:
random_gridCV = RandomizedSearchCV(pipeGB,param_distributions=params,scoring='r2',cv=kfold,n_jobs=-1)
random_gridCV.fit(X,y)
print(f'Best Params : {random_gridCV.best_params_}')
print(f'Best Score : {random_gridCV.best_score_}')

Best Params : {'GB__random_state': 26, 'GB__n_estimators': 100, 'GB__max_depth': 4, 'GB__learning_rate': np.float64(0.5044444444444445)}
Best Score : 0.9994548304120912


# catboost

In [63]:
cb = CatBoostRegressor()

In [65]:
pipeCB = Pipeline([("CT",ct),('CB',cb)])
pipeCB.get_params()

{'memory': None,
 'steps': [('CT',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000014B8990EAD0>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000014B8990ECF0>)],
                     verbose_feature_names_out=False)),
  ('CB', CatBoostRegressor(loss_function='RMSE'))],
 'transform_input': None,
 'verbose': False,
 'CT': ColumnTransformer(transformers=[('onehotencoder',
                                  OneHotEncoder(drop='first',
                                                sparse_output=False),
                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000014B899

In [66]:
params = {'CB__learning_rate':np.linspace(0.01,0.9,10),
          'CB__n_estimators':[25,50,100],
          'CB__max_depth': [None,2,3,4],
          'CB__random_state':[26]
         }

In [67]:
gridCV = GridSearchCV(pipeCB,param_grid=params,scoring='r2',cv=kfold,n_jobs=-1)
gridCV.fit(X,y)
print(f'Best Params : {gridCV.best_params_}')
print(f'Best Score : {gridCV.best_score_}')



C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:489: FitFailedWarning: 
11 fits failed out of a total of 600.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
11 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\ASUS\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 649, in fit
    self._final_estimator.fit(Xt

0:	learn: 63.3133746	total: 147ms	remaining: 14.5s
1:	learn: 37.9320266	total: 149ms	remaining: 7.28s
2:	learn: 27.7423426	total: 150ms	remaining: 4.85s
3:	learn: 22.5526535	total: 152ms	remaining: 3.64s
4:	learn: 19.6428285	total: 154ms	remaining: 2.92s
5:	learn: 16.5724363	total: 156ms	remaining: 2.44s
6:	learn: 15.4112179	total: 157ms	remaining: 2.09s
7:	learn: 14.2142210	total: 160ms	remaining: 1.84s
8:	learn: 13.3821332	total: 162ms	remaining: 1.64s
9:	learn: 12.9895592	total: 164ms	remaining: 1.48s
10:	learn: 12.2679461	total: 166ms	remaining: 1.35s
11:	learn: 11.9328800	total: 169ms	remaining: 1.24s
12:	learn: 11.2423210	total: 171ms	remaining: 1.14s
13:	learn: 10.3747553	total: 174ms	remaining: 1.07s
14:	learn: 9.9714111	total: 176ms	remaining: 996ms
15:	learn: 9.7172190	total: 179ms	remaining: 937ms
16:	learn: 9.2995784	total: 181ms	remaining: 885ms
17:	learn: 8.7019562	total: 184ms	remaining: 838ms
18:	learn: 8.2604516	total: 186ms	remaining: 794ms
19:	learn: 7.8151010	total:

In [70]:
random_gridCV = RandomizedSearchCV(pipeCB,param_distributions=params,scoring='r2',cv=kfold,n_jobs=-1)
random_gridCV.fit(X,y)
print(f'Best Params : {random_gridCV.best_params_}')
print(f'Best Score : {random_gridCV.best_score_}')

0:	learn: 63.3133746	total: 2.15ms	remaining: 213ms
1:	learn: 37.9320266	total: 4.76ms	remaining: 234ms
2:	learn: 27.7423426	total: 7.15ms	remaining: 231ms
3:	learn: 22.5526535	total: 9.69ms	remaining: 233ms
4:	learn: 19.6428285	total: 12.4ms	remaining: 235ms
5:	learn: 16.5724363	total: 14.9ms	remaining: 233ms
6:	learn: 15.4112179	total: 17.5ms	remaining: 232ms
7:	learn: 14.2142210	total: 20.1ms	remaining: 231ms
8:	learn: 13.3821332	total: 22.6ms	remaining: 228ms
9:	learn: 12.9895592	total: 24.9ms	remaining: 224ms
10:	learn: 12.2679461	total: 27.9ms	remaining: 225ms
11:	learn: 11.9328800	total: 30.5ms	remaining: 223ms
12:	learn: 11.2423210	total: 33.1ms	remaining: 222ms
13:	learn: 10.3747553	total: 35.8ms	remaining: 220ms
14:	learn: 9.9714111	total: 38.3ms	remaining: 217ms
15:	learn: 9.7172190	total: 40.8ms	remaining: 214ms
16:	learn: 9.2995784	total: 43.6ms	remaining: 213ms
17:	learn: 8.7019562	total: 46.5ms	remaining: 212ms
18:	learn: 8.2604516	total: 49ms	remaining: 209ms
19:	learn:

# LGBMRegressor

In [71]:
lgbm = LGBMRegressor()

In [72]:
pipeLGB = Pipeline([("CT",ct),('lgbm',lgbm)])
pipeLGB.get_params()

{'memory': None,
 'steps': [('CT',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000014B8990EAD0>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000014B8990ECF0>)],
                     verbose_feature_names_out=False)),
  ('lgbm', LGBMRegressor())],
 'transform_input': None,
 'verbose': False,
 'CT': ColumnTransformer(transformers=[('onehotencoder',
                                  OneHotEncoder(drop='first',
                                                sparse_output=False),
                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000014B8990EAD0>),
             

In [73]:
params = {'lgbm__learning_rate':np.linspace(0.01,0.9,10),
          'lgbm__n_estimators':[25,50,100],
          'lgbm__max_depth': [None,2,3,4],
          'lgbm__random_state':[26]
         }

In [74]:
gridCV = GridSearchCV(pipeLGB,param_grid=params,scoring='r2',cv=kfold,n_jobs=-1)
gridCV.fit(X,y)
print(f'Best Params : {gridCV.best_params_}')
print(f'Best Score : {gridCV.best_score_}')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000080 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 938
[LightGBM] [Info] Number of data points in the train set: 8645, number of used features: 11
[LightGBM] [Info] Start training from score 143.794448
Best Params : {'lgbm__learning_rate': np.float64(0.10888888888888888), 'lgbm__max_depth': None, 'lgbm__n_estimators': 100, 'lgbm__random_state': 26}
Best Score : 0.9995433869783042


In [76]:
random_gridCV = RandomizedSearchCV(pipeLGB,param_distributions=params,scoring='r2',cv=kfold,n_jobs=-1)
random_gridCV.fit(X,y)
print(f'Best Params : {random_gridCV.best_params_}')
print(f'Best Score : {random_gridCV.best_score_}')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000235 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 938
[LightGBM] [Info] Number of data points in the train set: 8645, number of used features: 11
[LightGBM] [Info] Start training from score 143.794448
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

# XGBRegressor

In [77]:
xg = XGBRegressor()

In [80]:
pipeXB = Pipeline([("CT",ct),('xg',xg)])
pipeXB.get_params()

{'memory': None,
 'steps': [('CT',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000014B8990EAD0>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000014B8990ECF0>)],
                     verbose_feature_names_out=False)),
  ('xg',
   XGBRegressor(base_score=None, booster=None, callbacks=None,
                colsample_bylevel=None, colsample_bynode=None,
                colsample_bytree=None, device=None, early_stopping_rounds=None,
                enable_categorical=True, eval_metric=None, feature_types=None,
                feature_weights=None, gamma=None, grow_policy=None,
                importance_type=None, i

In [81]:
params = {'xg__learning_rate':np.linspace(0.01,0.9,10),
          'xg__n_estimators':[25,50,100],
          'xg__max_depth': [None,2,3,4],
          'xg__random_state':[26]
         }

In [82]:
gridCV = GridSearchCV(pipeXB,param_grid=params,scoring='r2',cv=kfold,n_jobs=-1)
gridCV.fit(X,y)
print(f'Best Params : {gridCV.best_params_}')
print(f'Best Score : {gridCV.best_score_}')

Best Params : {'xg__learning_rate': np.float64(0.10888888888888888), 'xg__max_depth': None, 'xg__n_estimators': 100, 'xg__random_state': 26}
Best Score : 0.9996362328529358


In [83]:
random_gridCV = RandomizedSearchCV(pipeXB,param_distributions=params,scoring='r2',cv=kfold,n_jobs=-1)
random_gridCV.fit(X,y)
print(f'Best Params : {random_gridCV.best_params_}')
print(f'Best Score : {random_gridCV.best_score_}')

Best Params : {'xg__random_state': 26, 'xg__n_estimators': 100, 'xg__max_depth': 2, 'xg__learning_rate': np.float64(0.9)}
Best Score : 0.9995120286941528


# RandomForestRegressor

In [84]:
rf = RandomForestRegressor()

In [90]:
pipeRF = Pipeline([("CT",ct),('RF',rf)])
pipeRF.get_params()

{'memory': None,
 'steps': [('CT',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000014B8990EAD0>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x0000014B8990ECF0>)],
                     verbose_feature_names_out=False)),
  ('RF', RandomForestRegressor())],
 'transform_input': None,
 'verbose': False,
 'CT': ColumnTransformer(transformers=[('onehotencoder',
                                  OneHotEncoder(drop='first',
                                                sparse_output=False),
                                  <sklearn.compose._column_transformer.make_column_selector object at 0x0000014B8990EAD0>),
       

In [88]:
params = {'RF__max_features':[2,3,4,5],
          'RF__n_estimators':[25,50,100],
          'RF__max_depth': [None,2,3,4],
          'RF__random_state':[26]
         }

In [91]:
gridCV = GridSearchCV(pipeRF,param_grid=params,scoring='r2',cv=kfold,n_jobs=-1)
gridCV.fit(X,y)
print(f'Best Params : {gridCV.best_params_}')
print(f'Best Score : {gridCV.best_score_}')

Best Params : {'RF__max_depth': None, 'RF__max_features': 5, 'RF__n_estimators': 100, 'RF__random_state': 26}
Best Score : 0.9988735924728328


In [92]:
random_gridCV = RandomizedSearchCV(pipeRF,param_distributions=params,scoring='r2',cv=kfold,n_jobs=-1)
random_gridCV.fit(X,y)
print(f'Best Params : {random_gridCV.best_params_}')
print(f'Best Score : {random_gridCV.best_score_}')

Best Params : {'RF__random_state': 26, 'RF__n_estimators': 50, 'RF__max_features': 3, 'RF__max_depth': None}
Best Score : 0.9934963569698096
